# AzureML to Databricks model serving call

This notebook validates that an AzureML environment can authenticate to Databricks and call a model serving endpoint.

**Prereqs**
- Install packages if missing: `azure-identity`, `requests`.
- Ensure network connectivity from AzureML to Databricks (VNet/Private Link as needed).

**Required environment variables**
- `DATABRICKS_HOST` (e.g., `https://adb-<id>.<region>.azuredatabricks.net`)
- `DATABRICKS_ENDPOINT_NAME` (model serving endpoint name)

**Optional**
- `DATABRICKS_SAMPLE_INPUT_JSON` (JSON payload string; if omitted, a default payload is sent)

This uses AAD tokens (managed identity or service principal) for Databricks.

In [ ]:
import json
import os
import requests
import logging

# Latest Azure ML SDK v2
from azure.identity import ClientSecretCredential, DefaultAzureCredential, EnvironmentCredential, ChainedTokenCredential
from azure.core.exceptions import HttpResponseError

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

host = os.getenv("DATABRICKS_HOST")
endpoint_name = os.getenv("DATABRICKS_ENDPOINT_NAME")
if not all([host, endpoint_name]):
    raise ValueError("Missing DATABRICKS_HOST or DATABRICKS_ENDPOINT_NAME.")

tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

# Use ChainedTokenCredential for robust auth (latest pattern)
if all([tenant_id, client_id, client_secret]):
    credential = ChainedTokenCredential(
        EnvironmentCredential(),  # Service principal priority
        ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret),
        DefaultAzureCredential()  # Fallback
    )
    logger.info("Using service principal credentials for Databricks access")
else:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    logger.info("Using default credentials (managed identity or interactive)")

# Databricks resource scope (latest)
token = credential.get_token("2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default").token

endpoint_url = f"{host.rstrip('/')}/serving-endpoints/{endpoint_name}/invocations"

payload_text = os.getenv("DATABRICKS_SAMPLE_INPUT_JSON")
if payload_text:
    payload = json.loads(payload_text)
else:
    payload = {"dataframe_split": {"columns": ["text"], "data": [["hello from azureml"]]}}

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
try:
    response = requests.post(endpoint_url, headers=headers, json=payload, timeout=60)
    logger.info(f"Response status: {response.status_code}")
except Exception as e:
    logger.error(f"Request failed: {e}")
    raise

In [ ]:
# Optional: quick sanity check to confirm the token audience
print("Token acquired for Databricks scope.")